# Live Ingestion and Consolidation: Current-Season Integration with Historical Baseline

**Author:** [Your Name]
**Supervisor:** [Supervisor Name]
**Institution:** [Institution Name]
**Pipeline Stage:** 2 of 3 — Live Data Acquisition and Consolidation

---

## Abstract

This notebook performs the two production-cadence operations of the FPL
forecasting pipeline: (i) acquisition of the in-progress current season
via the official Fantasy Premier League REST API [1], and (ii)
consolidation of the resulting snapshot with the frozen multi-season
historical baseline produced by `historical_baseline.ipynb`.

Under the pipeline's separation-of-concerns design
(Option 1 architecture), this notebook is the only stage executed on a
per-gameweek cadence during the season. The historical baseline is
treated as an immutable input; live data is appended and reconciled,
and team-relative features — which by construction require access to
both historical and current-season rows in memory simultaneously — are
computed here over the combined dataset. The artefact produced,
`data/processed/master_training_set.csv`, is the direct input to
`forecasting_and_backtest.ipynb`.

## References

[1] Fantasy Premier League, "FPL API — bootstrap-static, fixtures, and
    event endpoints," Fantasy Premier League, London, UK, 2025.
    [Online]. Available: https://fantasy.premierleague.com/api/

[2] A. Vaastav, "Fantasy Premier League Historical Data," GitHub
    repository, 2024. [Online]. Available:
    https://github.com/vaastav/Fantasy-Premier-League

## 1. Introduction and Objectives

### 1.1 Motivation

Accurate per-gameweek point forecasts require a training set that
reflects the most recent real-world observations. The frozen historical
baseline of Notebook 1 covers seasons 2020-21 through 2024-25 but
necessarily excludes the current season, which is still in progress
at the time of prediction. A second-stage notebook is therefore required
that (a) acquires current-season performance data directly from the
FPL platform, (b) appends it to the historical base without corrupting
the latter's schema or identity system, and (c) computes aggregate
features that are only meaningful over the full dataset.

### 1.2 Problem Statement

Three issues dominate the design of this notebook:

1. **Live-API robustness.** The FPL API is unauthenticated, globally
   rate-limited, and occasionally intermittent. A naive loop of
   ``requests.get`` calls, as used in earlier prototype work, fails on
   transient errors and leaves partial artefacts on disk. A defensible
   implementation must incorporate explicit timeouts, retries with
   exponential back-off, and transaction-style persistence (write a
   tempfile, atomic rename).

2. **Dynamic gameweek discovery.** A hard-coded list of gameweeks
   (e.g. ``[29, 30, 31, 32]``) is not reproducible and silently drifts
   out of date. The ``events`` section of the ``bootstrap-static``
   endpoint reports each gameweek's lifecycle state; the set of
   gameweeks to ingest must be derived from these flags at runtime.

3. **Schema parity with the historical baseline.** Live API data and
   the historical CSVs share overlapping but non-identical column
   sets. Merging them without harmonisation produces a ragged table
   in which current-season rows carry ``NaN`` for columns present
   only in the historical data, and vice versa. Any such ragged
   shape implicitly teaches a downstream model that a missing value
   is a proxy for "this row came from the current season" — a form
   of data leakage that must be eliminated structurally.

### 1.3 Objectives

This notebook has four concrete deliverables:

- **O1.** Fetch the current season's static metadata (teams, players,
  fixtures) and per-gameweek live observations via a robust, retryable
  HTTP client.
- **O2.** Apply the same preprocessing contract as the historical
  notebook — schema harmonisation, fixture-derived team
  identification, position mapping, UUID resolution, absence flagging,
  rolling form — producing a current-season frame with exactly the
  historical baseline's schema.
- **O3.** Merge the current-season frame onto the historical baseline
  and resolve any overlap using a smart-deduplication rule that
  prefers the row with observed minutes over a zero-minute
  counterpart.
- **O4.** Compute team-relative features — per-gameweek contribution
  share, causal cumulative contribution share, and intra-team
  contribution rank — over the combined dataset, and apply the
  ghost-player filter that removes current-season entries with no
  recorded appearances.

### 1.4 Scope and Non-Goals

This notebook **does not** perform model training, hyperparameter
search, or backtesting — these are the responsibilities of
`forecasting_and_backtest.ipynb`. It also does not re-ingest
historical seasons from source CSVs; those are treated as a frozen
input from `data/processed/historical_baseline.csv`.

## 2. Methodology

### 2.1 Data Sources

| Source | Endpoint / Path | Used for |
| --- | --- | --- |
| FPL API — bootstrap | `GET /api/bootstrap-static/` | Teams, players, gameweek event states |
| FPL API — fixtures | `GET /api/fixtures/` | Match schedule with difficulties |
| FPL API — event live | `GET /api/event/{gw}/live/` | Per-player gameweek observations |
| Historical baseline | `data/processed/historical_baseline.csv` | Frozen training base from Notebook 1 |
| Player UUID map | `output/player_uuid_mapping.csv` | Cross-season player identity |

### 2.2 Robust HTTP Client

All API calls are routed through a single session-bound helper that
wraps `requests.Session` with:

- A connect-and-read timeout on every request.
- Retry-on-HTTP-error for the transient status codes
  $\{408, 429, 500, 502, 503, 504\}$ via `urllib3.util.Retry`.
- Exponential back-off between retries with a cap on total attempts.
- A descriptive `User-Agent` header identifying this as an academic
  research client.

### 2.3 Dynamic Gameweek Discovery

A gameweek is considered *ingestable* iff the bootstrap `events`
object reports it as `finished` **and** `data_checked`. The latter
flag indicates that the FPL platform has completed its post-match
data validation; ingesting before this has passed produces rows whose
stats are subsequently revised upstream, creating silent
inconsistencies between runs.

### 2.4 Schema Harmonisation with the Historical Baseline

The current-season frame is required to present an exact superset of
the historical baseline's columns. Where the live API omits a column
present in the historical baseline (e.g. `Clean Sheet`,
`Goals Conceded` — which are derivable from the team's match result
but not reported per-player in the live feed), the column is
synthesised from the fixture table. Where the historical baseline
lacks a column the API provides, the column is dropped from the
current-season frame. The result is two dataframes with byte-identical
column ordering, ready for concatenation.

### 2.5 Overlap Resolution via Minutes-Priority Deduplication

When a `(Player UUID, season, Gameweek, Opponent ID)` tuple is present
in both the historical baseline and the freshly-fetched current
season — which occurs for the current season's early gameweeks if
they were also present in the archive at the time the historical
baseline was built — the row with the greatest `Minutes Played` is
retained. Ties are broken by `Total Points`. This rule ensures that
the "richer" observation survives: a live API row reporting an
80-minute appearance takes precedence over a stale archive row that
recorded zero minutes, and vice versa.

### 2.6 Team Contribution Metrics

Let $p$ denote a player, $t \in p$ the team to which $p$ belongs,
and $w$ a gameweek. Let $\text{Pts}_{p,w}$ denote the player's FPL
points in gameweek $w$. The following features are computed over
the *combined* historical-plus-current dataset:

**Per-gameweek contribution share** —

$$\text{GW\%}_{p,t,w}
  \;=\; \frac{\text{Pts}_{p,w}}
             {\sum_{p' \in t}\,\text{Pts}_{p',w}} \times 100.$$

This expresses the player's share of their team's total points in
that gameweek; a value near 100 indicates that the player carried
the team's output.

**Causal cumulative contribution share** —

$$\text{Causal\%}_{p,t,w}
  \;=\; \frac{\sum_{w'<w} \text{Pts}_{p,w'}}
             {\sum_{w'<w}\sum_{p' \in t}\,\text{Pts}_{p',w'}} \times 100.$$

The strict inequality $w' < w$ makes this a *causal* feature: only
gameweeks strictly earlier than the current one are included, so the
feature is usable as a predictor without leakage. For $w=1$ the
feature is defined to be zero.

**Intra-team contribution rank** —

$$\text{Rank}_{p,t,w}
  \;=\; \operatorname{rank}_{\text{desc}}\bigl(\text{GW\%}_{p,t,w}
  \bigr)\bigm/\,|\,t\,|$$

computed within each $(t, w)$ group, with ties resolved by the
minimum-rank convention. The result lies in $(0, 1]$, with 1.0
indicating the team's top contributor for that gameweek.

### 2.7 Ghost-Player Filter

A "ghost" is defined as a player listed in the current season's
`players_raw.csv` but whose aggregate `Minutes Played` across the
season to date is zero. Such players inflate the frame with rows
that provide no training signal and never will (barring a future
appearance, in which case the filter is simply re-evaluated on
re-run). Ghosts are removed from the current-season partition only;
historical rows are left untouched.

### 2.8 Output Specification

The artefact is `data/processed/master_training_set.csv`. Its primary
key is the tuple
$(\text{Player UUID},\ \text{season},\ \text{Gameweek},\ \text{Opponent ID})$
— i.e. the same Double-Gameweek-safe key established in
`historical_baseline.ipynb`. Schema and dtypes are reported in
Section 5.

## 3. Environment and Dependencies

Imports are grouped by origin (standard library, third-party). A
structured `logging` configuration replaces ad-hoc `print` debugging,
paths are anchored to the project root via an explicit sentinel-file
search, and run-time configuration (current season identifier, API
base URL, timeout and retry policies) is centralised as named
constants.

In [112]:
"""Section 3.1 — Imports."""

# Standard library
from __future__ import annotations

import json
import logging
import re
import sys
import tempfile
import time
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable

# Third-party
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from unidecode import unidecode
from urllib3.util.retry import Retry

In [113]:
"""Section 3.2 — Logging configuration."""

import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger("live_ingestion_and_merge")
log.info("Logger initialised.")

15:07:33 | INFO    | Logger initialised.


In [114]:
"""Section 3.3 — Paths and run configuration."""


def _find_project_root(start: Path, sentinel: str = "master_team_list.csv") -> Path:
    """Locate the project root by walking upward to a sentinel file.

    Parameters
    ----------
    start : Path
        Directory from which to begin the upward walk.
    sentinel : str
        Filename whose presence marks the project root.

    Returns
    -------
    Path
        Resolved project root.

    Raises
    ------
    FileNotFoundError
        If the sentinel is not located before reaching the filesystem root.
    """
    current = start.resolve()
    while True:
        if (current / sentinel).exists():
            return current
        if current.parent == current:
            raise FileNotFoundError(
                f"Project root sentinel '{sentinel}' not found above {start}."
            )
        current = current.parent


PROJECT_ROOT = _find_project_root(Path.cwd())
DATA_ROOT = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "output"
MASTER_TEAM_LIST_PATH = PROJECT_ROOT / "master_team_list.csv"

# Run configuration — the only season-specific constant in the notebook.
CURRENT_SEASON: str = "2025-26"

# Input artefact (frozen output of Notebook 1).
HISTORICAL_BASELINE_PATH = PROCESSED_DIR / "historical_baseline.csv"

# Output artefact (consumed by Notebook 3).
MASTER_TRAINING_SET_PATH = PROCESSED_DIR / "master_training_set.csv"

# Per-season scratch directories populated by the live ingestion.
CURRENT_SEASON_DIR = DATA_ROOT / CURRENT_SEASON
CURRENT_SEASON_GW_DIR = CURRENT_SEASON_DIR / "gws"

# UUID mapping (shared with Notebook 1; extended here when new players appear).
UUID_MAPPING_PATH = OUTPUT_DIR / "player_uuid_mapping.csv"
CLEANED_UUID_MAPPING_PATH = OUTPUT_DIR / "player_uuid_mapping_cleaned.csv"

# API configuration.
FPL_API_BASE = "https://fantasy.premierleague.com/api"
HTTP_TIMEOUT_SECONDS = (10, 30)            # (connect, read)
HTTP_MAX_RETRIES = 5
HTTP_BACKOFF_FACTOR = 1.5
HTTP_RETRY_STATUSES = (408, 429, 500, 502, 503, 504)
HTTP_USER_AGENT = "fpl-thesis-pipeline/1.0 (academic research client)"

# Domain constants shared with Notebook 1.
POSITION_MAP: dict[int, str] = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}

ROLLING_METRICS: tuple[str, ...] = (
    "Total Points", "Minutes Played", "Goals Scored", "Assists",
    "Goals Conceded", "ICT Index", "Threat", "Creativity", "Influence",
)
ROLLING_WINDOWS: tuple[int, ...] = (3, 5)
ABSENCE_STREAK_THRESHOLD = 3

# Ensure scratch directories exist.
for d in (PROCESSED_DIR, OUTPUT_DIR, CURRENT_SEASON_DIR, CURRENT_SEASON_GW_DIR):
    d.mkdir(parents=True, exist_ok=True)

log.info("Project root: %s", PROJECT_ROOT)
log.info("Current season: %s", CURRENT_SEASON)
log.info("Historical baseline: %s", HISTORICAL_BASELINE_PATH)
log.info("Master training set: %s", MASTER_TRAINING_SET_PATH)

15:07:33 | INFO    | Project root: C:\Python\fpl_pipeline
15:07:33 | INFO    | Current season: 2025-26
15:07:33 | INFO    | Historical baseline: C:\Python\fpl_pipeline\data\processed\historical_baseline.csv
15:07:33 | INFO    | Master training set: C:\Python\fpl_pipeline\data\processed\master_training_set.csv


## 4. Implementation

As in Notebook 1, the pipeline is decomposed into single-responsibility
functions composed by an orchestration cell at the end of this section.

Sections 4.1–4.4 concern live-API acquisition. Sections 4.5–4.9
perform the same preprocessing contract as Notebook 1 — restated here
rather than imported — to keep the notebook self-contained and
readable end-to-end by a reader who has seen only this file. Sections
4.10–4.13 perform the merge, the team-contribution feature
construction, the ghost-player filter, and the final projection.

In [115]:
"""Section 4.1 — Robust HTTP session factory."""


def build_http_session() -> requests.Session:
    """Return a ``requests.Session`` hardened for the FPL API.

    The session is configured with a retry strategy that handles the
    transient HTTP statuses listed in ``HTTP_RETRY_STATUSES`` via
    exponential back-off, applies ``HTTP_TIMEOUT_SECONDS`` to every
    request (enforced in :func:`get_json`), and advertises an academic
    ``User-Agent``.

    Returns
    -------
    requests.Session
        Configured session.
    """
    session = requests.Session()
    session.headers.update({"User-Agent": HTTP_USER_AGENT})

    retry = Retry(
        total=HTTP_MAX_RETRIES,
        connect=HTTP_MAX_RETRIES,
        read=HTTP_MAX_RETRIES,
        status=HTTP_MAX_RETRIES,
        status_forcelist=HTTP_RETRY_STATUSES,
        backoff_factor=HTTP_BACKOFF_FACTOR,
        allowed_methods=frozenset({"GET"}),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=4, pool_maxsize=8)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def get_json(session: requests.Session, url: str) -> Any:
    """Fetch ``url`` via ``session`` and decode the JSON response.

    Parameters
    ----------
    session : requests.Session
        A session built by :func:`build_http_session`.
    url : str
        Absolute URL to GET.

    Returns
    -------
    object
        Parsed JSON payload.

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status after retries exhaust.
    """
    response = session.get(url, timeout=HTTP_TIMEOUT_SECONDS)
    response.raise_for_status()
    return response.json()


HTTP_SESSION = build_http_session()

In [116]:
"""Section 4.2 — Static-data acquisition (teams, players, fixtures)."""


def _atomic_write_csv(df: pd.DataFrame, destination: Path) -> None:
    """Write ``df`` to ``destination`` via a temporary file + rename.

    This avoids leaving a half-written CSV on disk if the process is
    interrupted mid-write — a correctness-critical property for a
    pipeline that may be executed unattended during a defense.
    """
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".csv", dir=destination.parent, delete=False, encoding="utf-8-sig"
    ) as tmp:
        df.to_csv(tmp.name, index=False)
        tmp_path = Path(tmp.name)
    tmp_path.replace(destination)


def fetch_static_data(session: requests.Session) -> list[dict]:
    """Fetch teams, players and fixtures; persist them to ``CURRENT_SEASON_DIR``.

    Parameters
    ----------
    session : requests.Session
        Hardened session from :func:`build_http_session`.

    Returns
    -------
    list[dict]
        The ``events`` section of the bootstrap payload, used downstream
        by :func:`discover_ingestable_gameweeks`.
    """
    log.info("Fetching bootstrap-static and fixtures …")

    bootstrap = get_json(session, f"{FPL_API_BASE}/bootstrap-static/")
    fixtures_payload = get_json(session, f"{FPL_API_BASE}/fixtures/")

    # Teams
    teams_df = pd.DataFrame(
        [{"id": t["id"], "name": t["name"], "short_name": t["short_name"]}
         for t in bootstrap["teams"]]
    )
    _atomic_write_csv(teams_df, CURRENT_SEASON_DIR / "teams.csv")

    # Players (retain same column subset used by Notebook 1's loader).
    players_df = pd.DataFrame(bootstrap["elements"])
    keep_cols = ["id", "team", "element_type", "first_name", "second_name", "web_name"]
    players_out = players_df.loc[:, keep_cols].rename(columns={"id": "element"})
    _atomic_write_csv(players_out, CURRENT_SEASON_DIR / "players_raw.csv")

    # Fixtures
    fixtures_df = pd.DataFrame(fixtures_payload)
    keep_cols_fx = [
        "id", "event", "team_h", "team_a",
        "team_h_difficulty", "team_a_difficulty",
        "team_h_score", "team_a_score",
        "kickoff_time",
    ]
    present = [c for c in keep_cols_fx if c in fixtures_df.columns]
    _atomic_write_csv(fixtures_df.loc[:, present], CURRENT_SEASON_DIR / "fixtures.csv")

    log.info(
        "Static data written: %d teams, %d players, %d fixtures.",
        len(teams_df), len(players_out), len(fixtures_df),
    )
    return bootstrap["events"]

In [117]:
"""Section 4.3 — Dynamic gameweek discovery."""


def discover_ingestable_gameweeks(events: list[dict]) -> list[int]:
    """Return the list of gameweek ids whose data is safe to ingest.

    A gameweek is ingestable iff both ``finished`` and ``data_checked``
    are ``True`` in its bootstrap event object. The list is returned
    in ascending order.

    Parameters
    ----------
    events : list[dict]
        The ``events`` section of the bootstrap payload.

    Returns
    -------
    list[int]
        Sorted list of gameweek ids.
    """
    ingestable = sorted(
        int(e["id"]) for e in events
        if e.get("finished") and e.get("data_checked")
    )
    if not ingestable:
        log.warning(
            "No gameweek is simultaneously finished and data-checked. "
            "Has the season started yet?"
        )
    else:
        log.info(
            "Ingestable gameweeks (finished & data-checked): %s",
            ingestable,
        )
    return ingestable

In [118]:
"""Section 4.4 — Per-gameweek live data acquisition."""

LIVE_STAT_KEEP: tuple[str, ...] = (
    "element", "minutes", "goals_scored", "assists", "clean_sheets",
    "goals_conceded", "yellow_cards", "red_cards", "total_points",
    "influence", "creativity", "threat", "ict_index",
)


def fetch_live_gameweek(
    session: requests.Session,
    gw: int,
    players_raw: pd.DataFrame,
    fixtures: pd.DataFrame,
    season: str,
) -> pd.DataFrame:
    """Fetch and assemble a single gameweek's live frame.

    The FPL ``event/{gw}/live/`` endpoint returns a row for *every*
    registered player regardless of whether their team had a fixture
    in that gameweek. Players whose team did not play (postponed
    matches, non-rostered loans, etc.) would otherwise pollute the
    frame with no-opponent, no-team rows. Such rows are dropped here
    via an inner-join on the opponent table.

    Parameters
    ----------
    session : requests.Session
        Hardened session.
    gw : int
        Gameweek number.
    players_raw : pandas.DataFrame
        Players table written by :func:`fetch_static_data`.
    fixtures : pandas.DataFrame
        Fixtures table written by :func:`fetch_static_data`.
    season : str
        Season identifier.

    Returns
    -------
    pandas.DataFrame
        One row per (player, fixture-they-played-in) for the gameweek.
    """
    payload = get_json(session, f"{FPL_API_BASE}/event/{gw}/live/")

    rows: list[dict] = []
    for elem in payload["elements"]:
        stats = dict(elem["stats"])
        stats["element"] = elem["id"]
        rows.append(stats)

    base = pd.DataFrame(rows)
    keep_present = [c for c in LIVE_STAT_KEEP if c in base.columns]
    base = base.loc[:, keep_present].copy()
    base["Gameweek"] = gw
    base["season"] = season

    players_local = players_raw.copy()
    players_local["name"] = (
        players_local["first_name"].fillna("") + " " + players_local["second_name"].fillna("")
    ).str.strip()
    base = base.merge(
        players_local[["element", "name", "team"]],
        on="element", how="left",
    )

    gw_fixtures = fixtures[fixtures["event"] == gw]
    opp_rows: list[dict] = []
    for _, fx in gw_fixtures.iterrows():
        opp_rows.append({"team": int(fx["team_h"]), "opponent_team": int(fx["team_a"]), "was_home": True})
        opp_rows.append({"team": int(fx["team_a"]), "opponent_team": int(fx["team_h"]), "was_home": False})
    opp_df = pd.DataFrame(opp_rows)

    # Inner join — players whose team had no fixture this GW (postponed,
    # loaned out, non-rostered) are excluded. This is correct: a "no
    # match" row carries no training signal.
    n_before = len(base)
    combined = base.merge(opp_df, on="team", how="inner")
    n_dropped = n_before - combined["element"].nunique()
    if n_dropped > 0:
        log.info(
            "GW %d: dropped %d players whose team had no fixture.",
            gw, n_dropped,
        )
    return combined

### 4.5 Shared Preprocessing Library

The next four cells reproduce the preprocessing contract already
established in `historical_baseline.ipynb`: name normalisation,
fixture-derived team identification, canonical schema enforcement,
position repair, vectorised absence flagging, and leakage-safe rolling
features. They are restated verbatim here so that this notebook is
self-contained and can be read without reference to Notebook 1.

A reader comparing the two notebooks will find these sections
byte-identical with those of Notebook 1; this is deliberate — the
two pipeline stages are required to apply *exactly* the same
transformations so that their outputs are schema-compatible.

In [119]:
"""Section 4.5.1 — Player-name normalisation (shared with Notebook 1)."""


def normalize_player_name(name: object) -> object:
    """Return a canonical, comparable representation of a player name.

    See :func:`historical_baseline.normalize_player_name` for the full
    transformation chain; this implementation is identical.
    """
    if pd.isna(name):
        return name
    text = str(name).strip().lower()
    text = unidecode(text)
    text = re.sub(r"[\s_]*\d+\s*$", "", text)
    text = text.replace("_", " ")
    text = "".join(ch for ch in text if ch.isalnum() or ch.isspace())
    text = " ".join(text.split())
    return text

In [120]:
"""Section 4.5.2 — Fixture-derived team identification (shared with Notebook 1)."""


def _build_fixture_long_current(season: str) -> pd.DataFrame:
    """Long-format fixture table for the current season."""
    fx_raw = pd.read_csv(CURRENT_SEASON_DIR / "fixtures.csv")
    gw_col = "event" if "event" in fx_raw.columns else "round"
    fx_raw[gw_col] = pd.to_numeric(fx_raw[gw_col], errors="coerce")
    fx_raw = fx_raw.dropna(subset=[gw_col, "team_h", "team_a"])

    rows: list[dict] = []
    for _, r in fx_raw.iterrows():
        gw = int(r[gw_col])
        h, a = int(r["team_h"]), int(r["team_a"])
        dh = int(r["team_h_difficulty"]) if pd.notna(r["team_h_difficulty"]) else 0
        da = int(r["team_a_difficulty"]) if pd.notna(r["team_a_difficulty"]) else 0
        rows.append({
            "Gameweek": gw, "OppKey": a,
            "Player Team ID": h, "Opponent ID": a,
            "Is Home": True, "Opponent Difficulty": dh,
        })
        rows.append({
            "Gameweek": gw, "OppKey": h,
            "Player Team ID": a, "Opponent ID": h,
            "Is Home": False, "Opponent Difficulty": da,
        })

    long = pd.DataFrame(rows)
    long["Gameweek"] = long["Gameweek"].astype("Int64")
    long["OppKey"] = long["OppKey"].astype("Int64")
    return long


def build_current_season_core(season: str) -> pd.DataFrame:
    """Load and assemble the current season's GW CSVs into a core frame.

    Parameters
    ----------
    season : str
        Current season identifier.

    Returns
    -------
    pandas.DataFrame
        Frame enriched with player metadata, fixture-derived team
        identification, human-readable team names and composed
        ``Player Name`` / ``Web Name`` columns — matching the structure
        produced by Notebook 1's ``derive_team_from_fixtures``.
    """
    log.info("Assembling current-season core frame from per-GW files …")

    gw_files = sorted(
        CURRENT_SEASON_GW_DIR.glob("gw*.csv"),
        key=lambda p: int(p.stem.replace("gw", "")),
    )
    if not gw_files:
        raise FileNotFoundError(
            f"No gameweek CSVs present under {CURRENT_SEASON_GW_DIR}."
        )
    gw_frames = [pd.read_csv(p) for p in gw_files]
    df = pd.concat(gw_frames, ignore_index=True)

    # Coerce identifier dtypes.
    for c in ("element", "Gameweek", "opponent_team", "team"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    # Player metadata (intentionally excludes live ``team`` — team
    # identity comes from fixtures, not ``players_raw``).
    players_raw = pd.read_csv(CURRENT_SEASON_DIR / "players_raw.csv")
    players_raw["element"] = pd.to_numeric(players_raw["element"], errors="coerce").astype("Int64")
    meta_cols = [c for c in ("element", "element_type", "web_name", "first_name", "second_name")
                 if c in players_raw.columns]
    df = df.merge(players_raw[meta_cols], on="element", how="left")

    # Fixture-derived team identification.
    fixture_long = _build_fixture_long_current(season)
    df = df.merge(
        fixture_long,
        left_on=["Gameweek", "opponent_team"],
        right_on=["Gameweek", "OppKey"],
        how="left",
    ).drop(columns=["OppKey"])

    # Human-readable team names.
    teams = pd.read_csv(CURRENT_SEASON_DIR / "teams.csv").rename(
        columns={"id": "Team ID", "name": "Team Name"}
    )
    teams["Team ID"] = pd.to_numeric(teams["Team ID"], errors="coerce").astype("Int64")
    df = df.merge(
        teams[["Team ID", "Team Name"]],
        left_on="Player Team ID", right_on="Team ID", how="left",
    ).rename(columns={"Team Name": "Player Team Name"}).drop(columns=["Team ID"])
    df = df.merge(
        teams.rename(columns={"Team ID": "Opponent ID", "Team Name": "Opponent Name"})[
            ["Opponent ID", "Opponent Name"]
        ],
        on="Opponent ID", how="left",
    )

    # Composite Player Name (falls back to raw 'name' when first/second are missing).
    if {"first_name", "second_name"}.issubset(df.columns):
        composed = (
            df["first_name"].fillna("") + " " + df["second_name"].fillna("")
        ).str.strip().replace("", np.nan)
        df["Player Name"] = composed.fillna(df["name"])
    else:
        df["Player Name"] = df["name"]
    df["Web Name"] = df.get("web_name")

    return df

In [121]:
"""Section 4.5.3 — Canonical schema and position repair (shared with Notebook 1)."""

CANONICAL_RENAME: dict[str, str] = {
    "element": "Code",
    "minutes": "Minutes Played",
    "goals_scored": "Goals Scored",
    "assists": "Assists",
    "clean_sheets": "Clean Sheet",
    "goals_conceded": "Goals Conceded",
    "yellow_cards": "Yellow Card",
    "red_cards": "Red Cards",
    "total_points": "Total Points",
    "influence": "Influence",
    "creativity": "Creativity",
    "threat": "Threat",
    "ict_index": "ICT Index",
}


def assign_canonical_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Apply canonical column names and derive the ``Position`` column."""
    out = df.rename(columns=CANONICAL_RENAME).copy()
    if "element_type" in out.columns:
        out["Position"] = out["element_type"].map(POSITION_MAP)
    else:
        out["Position"] = np.nan
    out["Player Name Norm"] = out["Player Name"].apply(normalize_player_name)
    return out


def repair_missing_positions_from_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """Back-fill missing ``Position`` using the current season's players_raw."""
    if df["Position"].isna().sum() == 0:
        return df
    players_raw = pd.read_csv(CURRENT_SEASON_DIR / "players_raw.csv")
    players_raw["element"] = pd.to_numeric(players_raw["element"], errors="coerce").astype("Int64")
    code_to_type = dict(zip(players_raw["element"], players_raw["element_type"]))

    out = df.copy()
    mask = out["Position"].isna()
    out.loc[mask, "Position"] = out.loc[mask, "Code"].map(code_to_type).map(POSITION_MAP)

    remaining = int(out["Position"].isna().sum())
    log.info("Position repair: %d rows still unresolved.", remaining)
    return out

In [122]:
"""Section 4.5.4 — Source-CSV dedup + DGW indexing (shared with Notebook 1)."""


def deduplicate_and_index_fixtures(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse source duplicates and assign ``Fixture Index``.

    See §4.7b of Notebook 1 for the full rationale. The composite key
    ``(Code, season, Gameweek, Opponent ID)`` is unique per real match;
    where rows collide on this key the one with the greatest
    ``Minutes Played`` is retained (ties broken by ``Total Points``).
    """
    n_before = len(df)
    work = df.copy()
    work["_minutes_num"] = pd.to_numeric(work["Minutes Played"], errors="coerce").fillna(-1)
    work["_points_num"] = pd.to_numeric(work["Total Points"], errors="coerce").fillna(-1)
    work["_tiebreak"] = np.arange(len(work))

    key = ["Code", "season", "Gameweek", "Opponent ID"]
    work = work.sort_values(
        by=key + ["_minutes_num", "_points_num", "_tiebreak"],
        ascending=[True, True, True, True, False, False, True],
        kind="mergesort",
    )
    deduped = work.drop_duplicates(subset=key, keep="first").drop(
        columns=["_minutes_num", "_points_num", "_tiebreak"]
    )

    n_dropped = n_before - len(deduped)
    if n_dropped > 0:
        log.info("Dropped %s source-CSV duplicate rows.", f"{n_dropped:,}")

    deduped = deduped.sort_values(
        ["Code", "season", "Gameweek", "Opponent ID"], kind="mergesort"
    )
    deduped["Fixture Index"] = (
        deduped.groupby(["Code", "season", "Gameweek"]).cumcount() + 1
    ).astype("Int64")
    return deduped.reset_index(drop=True)

In [123]:
"""Section 4.5.5 — UUID identity pipeline (shared with Notebook 1)."""


def assign_player_uuids(
    df: pd.DataFrame,
    cleaned_path: Path = CLEANED_UUID_MAPPING_PATH,
    auto_path: Path = UUID_MAPPING_PATH,
) -> pd.DataFrame:
    """Attach a stable ``Player UUID`` (two-tier curated + auto mapping).

    This implementation is identical to Notebook 1 §4.10; the curated
    file is never modified, while the auto file is read, extended with
    fresh UUIDs for previously-unseen normalised names, and rewritten.
    """
    out = df.copy()

    curated_lookup: dict[str, str] = {}
    if cleaned_path.exists():
        curated = pd.read_csv(cleaned_path)
        norm_col = next(c for c in curated.columns if c.lower() == "player name norm")
        uuid_col = next(c for c in curated.columns if c.lower() == "player uuid")
        curated_lookup = dict(
            zip(curated[norm_col].astype(str), curated[uuid_col].astype(str))
        )
        log.info("Loaded %d curated UUID entries.", len(curated_lookup))

    auto_lookup: dict[str, str] = {}
    if auto_path.exists():
        existing = pd.read_csv(auto_path)
        if {"Player Name Norm", "Player UUID"}.issubset(existing.columns):
            auto_lookup = dict(
                zip(existing["Player Name Norm"].astype(str), existing["Player UUID"].astype(str))
            )
        log.info("Loaded %d existing UUID entries.", len(auto_lookup))

    newly_minted = 0
    for name in sorted(out["Player Name Norm"].dropna().astype(str).unique()):
        if name in curated_lookup:
            continue
        if name not in auto_lookup:
            auto_lookup[name] = str(uuid.uuid4())
            newly_minted += 1
    if newly_minted:
        log.info("Minted %d new UUIDs for previously-unseen names.", newly_minted)

    representative = (
        out.dropna(subset=["Player Name Norm"])
        .groupby("Player Name Norm")[["Player Name", "Web Name"]]
        .agg(lambda s: s.dropna().iloc[0] if not s.dropna().empty else np.nan)
        .reset_index()
    )
    persisted = (
        pd.DataFrame({
            "Player Name Norm": list(auto_lookup.keys()),
            "Player UUID": list(auto_lookup.values()),
        })
        .merge(representative, on="Player Name Norm", how="left")
    )
    auto_path.parent.mkdir(parents=True, exist_ok=True)
    persisted.to_csv(auto_path, index=False, encoding="utf-8-sig")
    log.info("Persisted %d total UUID entries.", len(persisted))

    final_lookup = {**auto_lookup, **curated_lookup}
    out["Player UUID"] = out["Player Name Norm"].map(final_lookup)
    return out


def reconcile_uuids_via_code(
    df: pd.DataFrame,
    auto_path: Path = UUID_MAPPING_PATH,
) -> pd.DataFrame:
    """Merge UUIDs that co-occur under the same ``(season, Code)``."""
    out = df.copy()
    pairs = (
        out.dropna(subset=["Player UUID", "Code"])
        .groupby(["season", "Code"])["Player UUID"]
        .unique()
    )
    parent: dict[str, str] = {}

    def find(x: str) -> str:
        while parent.get(x, x) != x:
            parent[x] = parent.get(parent[x], parent[x])
            x = parent[x]
        return x

    def union(a: str, b: str) -> None:
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        winner, loser = (ra, rb) if ra < rb else (rb, ra)
        parent[loser] = winner

    for uuid_set in pairs:
        if len(uuid_set) < 2:
            continue
        canonical = min(uuid_set)
        for other in uuid_set:
            if other != canonical:
                union(canonical, other)

    all_uuids = out["Player UUID"].dropna().astype(str).unique()
    rewrite = {u: find(u) if u in parent else u for u in all_uuids}
    n_merged = sum(1 for u, v in rewrite.items() if u != v)
    if n_merged == 0:
        return out

    log.info("UUID reconciliation: collapsing %d aliased UUIDs.", n_merged)
    out["Player UUID"] = out["Player UUID"].map(rewrite).fillna(out["Player UUID"])

    if auto_path.exists():
        persisted = pd.read_csv(auto_path)
        if "Player UUID" in persisted.columns:
            persisted["Player UUID"] = (
                persisted["Player UUID"].map(rewrite).fillna(persisted["Player UUID"])
            )
            persisted = persisted.drop_duplicates(subset=["Player Name Norm"], keep="first")
            persisted.to_csv(auto_path, index=False, encoding="utf-8-sig")
    return out


def split_homonym_uuids(
    df: pd.DataFrame,
    auto_path: Path = UUID_MAPPING_PATH,
) -> pd.DataFrame:
    """Split UUIDs that collapse distinct FPL ``Code`` values (see Notebook 1 §4.10c)."""
    out = df.copy()
    non_null = out.dropna(subset=["Player UUID", "Code"])
    uuid_to_codes = non_null.groupby("Player UUID")["Code"].unique()
    offenders = uuid_to_codes[uuid_to_codes.apply(len) > 1]
    if offenders.empty:
        return out

    log.warning("Homonym split: %d UUID(s) span multiple FPL Codes.", len(offenders))
    new_mapping_rows: list[dict] = []

    for original_uuid, _codes in offenders.items():
        subset = non_null[non_null["Player UUID"] == original_uuid]
        code_counts = subset["Code"].value_counts()
        minority_codes = code_counts.index[1:].tolist()
        for rank, minor_code in enumerate(minority_codes, start=1):
            new_uuid = str(uuid.uuid4())
            mask = (out["Player UUID"] == original_uuid) & (out["Code"] == minor_code)
            representative = out.loc[mask].iloc[0]
            disambiguated_norm = f"{representative['Player Name Norm']}#{rank}"
            out.loc[mask, "Player UUID"] = new_uuid
            out.loc[mask, "Player Name Norm"] = disambiguated_norm
            new_mapping_rows.append({
                "Player Name Norm": disambiguated_norm,
                "Player UUID": new_uuid,
                "Player Name": representative["Player Name"],
                "Web Name": representative.get("Web Name"),
            })

    if new_mapping_rows and auto_path.exists():
        existing = pd.read_csv(auto_path)
        augmented = pd.concat([existing, pd.DataFrame(new_mapping_rows)], ignore_index=True)
        augmented = augmented.drop_duplicates(subset=["Player Name Norm"], keep="first")
        augmented.to_csv(auto_path, index=False, encoding="utf-8-sig")
    return out

In [124]:
"""Section 4.5.6 — Absence flag and rolling features (shared with Notebook 1)."""


def flag_consecutive_absences(
    df: pd.DataFrame,
    threshold: int = ABSENCE_STREAK_THRESHOLD,
) -> pd.DataFrame:
    """Add ``Injury/Unavailable`` — vectorised consecutive-zero-minute flag."""
    out = df.sort_values(["Player UUID", "season", "Gameweek"]).copy()
    minutes = pd.to_numeric(out["Minutes Played"], errors="coerce").fillna(0)
    is_zero = (minutes == 0).astype(int)
    run_break = (minutes != 0).astype(int)
    run_id = run_break.groupby(out["Player UUID"]).cumsum()
    streak = is_zero.groupby([out["Player UUID"], run_id]).cumsum()
    out["Injury/Unavailable"] = (streak >= threshold).astype("int8")
    return out


def compute_rolling_features(
    df: pd.DataFrame,
    metrics: Iterable[str] = ROLLING_METRICS,
    windows: Iterable[int] = ROLLING_WINDOWS,
) -> pd.DataFrame:
    """Append leakage-safe rolling-mean features per (UUID, season)."""
    out = df.sort_values(["Player UUID", "season", "Gameweek"]).copy()
    grouper = out.groupby(["Player UUID", "season"], sort=False)
    for window in windows:
        for metric in metrics:
            if metric not in out.columns:
                continue
            out[f"Avg_{metric}_L{window}"] = (
                grouper[metric]
                .transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
                .fillna(0.0)
            )
    return out

In [125]:
"""Section 4.6 — Current-season synthesis.

Composes the live-ingestion and preprocessing functions above into a
single call that produces a fully-featured current-season frame with
exactly the historical baseline's schema (modulo the rolling features,
which are re-computed downstream over the combined dataset).
"""


def build_current_season_frame() -> pd.DataFrame:
    """Produce the fully-featured current-season frame.

    Returns
    -------
    pandas.DataFrame
        Current-season player–gameweek frame, schema-compatible with
        ``historical_baseline.csv``.
    """
    events = fetch_static_data(HTTP_SESSION)
    gameweeks = discover_ingestable_gameweeks(events)
    if gameweeks:
        fetch_all_live_gameweeks(HTTP_SESSION, gameweeks, season=CURRENT_SEASON)
    else:
        log.warning("Current season has no data-checked gameweeks yet.")

    core = build_current_season_core(CURRENT_SEASON)
    core = assign_canonical_schema(core)
    core = repair_missing_positions_from_metadata(core)
    core = deduplicate_and_index_fixtures(core)
    core = assign_player_uuids(core)
    core = reconcile_uuids_via_code(core)
    core = split_homonym_uuids(core)
    log.info("Current-season frame: %s rows built.", f"{len(core):,}")
    return core

In [126]:
"""Section 4.7 — Overlap resolution (minutes-priority deduplication)."""


def merge_with_historical(
    historical: pd.DataFrame,
    current: pd.DataFrame,
) -> pd.DataFrame:
    """Concatenate historical + current and resolve overlap.

    Rows that collide on ``(Player UUID, season, Gameweek, Opponent ID)``
    — which occurs only if the historical baseline happens to include
    rows for ``CURRENT_SEASON`` that are also now reported by the live
    API — are resolved by preferring the row with the greatest
    ``Minutes Played`` (ties by ``Total Points``).

    Parameters
    ----------
    historical : pandas.DataFrame
        Frozen historical baseline from Notebook 1.
    current : pandas.DataFrame
        Current-season frame from :func:`build_current_season_frame`.

    Returns
    -------
    pandas.DataFrame
        Combined, overlap-resolved frame.
    """
    # Align schemas: take the union of columns, fill missing with NA.
    all_cols = list(
        dict.fromkeys(list(historical.columns) + list(current.columns))
    )
    hist_aligned = historical.reindex(columns=all_cols)
    curr_aligned = current.reindex(columns=all_cols)

    combined = pd.concat([hist_aligned, curr_aligned], ignore_index=True)

    n_before = len(combined)
    work = combined.copy()
    work["_minutes_num"] = pd.to_numeric(work["Minutes Played"], errors="coerce").fillna(-1)
    work["_points_num"] = pd.to_numeric(work["Total Points"], errors="coerce").fillna(-1)
    work["_tiebreak"] = np.arange(len(work))

    key = ["Player UUID", "season", "Gameweek", "Opponent ID"]
    work = work.sort_values(
        by=key + ["_minutes_num", "_points_num", "_tiebreak"],
        ascending=[True, True, True, True, False, False, True],
        kind="mergesort",
    )
    resolved = work.drop_duplicates(subset=key, keep="first").drop(
        columns=["_minutes_num", "_points_num", "_tiebreak"]
    )
    n_resolved = n_before - len(resolved)
    log.info(
        "Merge complete: %s historical + %s current → %s combined "
        "(%d overlap rows resolved).",
        f"{len(historical):,}", f"{len(current):,}", f"{len(resolved):,}", n_resolved,
    )
    return resolved.sort_values(key).reset_index(drop=True)

In [127]:
"""Section 4.8 — Team contribution metrics.

Implements the three features formally defined in §2.6: per-gameweek
share, causal cumulative share, and intra-team rank, computed over
the *combined* historical-plus-current frame on the integer
``Player Team ID`` key.

**Double-Gameweek handling.** Team totals are computed by summing
per-player points within each (Team ID, season, Gameweek) group;
the sum over each team's DGW players correctly includes both
fixtures by construction. When building cumulative team series, one
representative row per (Team ID, season, Gameweek) group is taken —
this avoids the "DGW inflation" bug in which a naive cumsum over
every row multiplies the team total by the number of team rows.

**Negative-points handling.** A player may score negative FPL
points in a single gameweek (penalty miss: −2; own goal: −2;
red card: −3). For the contribution aggregation, negative
per-player points are floored at zero *for the feature computation
only*; the raw ``Total Points`` column is preserved unchanged. The
rationale is that a contribution share is mathematically a share of
a non-negative whole.
"""


def compute_team_contributions(df: pd.DataFrame) -> pd.DataFrame:
    """Add the four team-contribution features.

    Parameters
    ----------
    df : pandas.DataFrame
        Combined frame containing ``Player UUID``, ``Player Team ID``,
        ``season``, ``Gameweek`` and ``Total Points``.

    Returns
    -------
    pandas.DataFrame
        Input frame augmented with:
          - ``Team Total Points GW``
          - ``Team GW Contribution Pct``
          - ``Team Causal Contribution Pct``
          - ``Team Contribution Rank GW``
    """
    out = df.copy()
    if "Player Team ID" not in out.columns:
        raise KeyError(
            "compute_team_contributions requires 'Player Team ID'. "
            "Ensure both Notebooks 1 and 2 include it in BASE_COLUMNS."
        )

    team_col = "Player Team ID"
    tgw_cols = [team_col, "season", "Gameweek"]

    # Non-negative per-player points for the *feature* computation.
    # The raw Total Points column is NOT modified.
    pts_nn = (
        pd.to_numeric(out["Total Points"], errors="coerce")
        .fillna(0)
        .clip(lower=0)
    )
    out["_pts_nn"] = pts_nn

    # --- Team total in the gameweek (summed across all rows of that team-GW,
    #     which correctly accumulates both DGW fixtures' points).
    out["Team Total Points GW"] = (
        out.groupby(tgw_cols)["_pts_nn"].transform("sum").astype(float)
    )

    # --- Per-gameweek contribution share.
    with np.errstate(divide="ignore", invalid="ignore"):
        gw_pct = np.where(
            out["Team Total Points GW"] > 0,
            out["_pts_nn"] / out["Team Total Points GW"] * 100.0,
            0.0,
        )
    out["Team GW Contribution Pct"] = np.round(gw_pct, 2)

    # --- Intra-team rank within the gameweek (percentile; 1.0 = top).
    out["Team Contribution Rank GW"] = (
        out.groupby(tgw_cols)["_pts_nn"]
        .rank(method="min", ascending=False, pct=True)
        .round(4)
    )

    # --- Causal cumulative share.
    # Step 1: Build one row per (Team ID, season, GW) with the team total,
    # then compute strict-prior cumulative. This avoids the DGW-inflation
    # bug: if we cumsum a per-row column that has the same value repeated
    # for every team member (and every fixture), the cumulative balloons
    # proportionally to the team size × DGW multiplicity.
    team_totals = (
        out.drop_duplicates(subset=tgw_cols)[tgw_cols + ["Team Total Points GW"]]
        .sort_values(tgw_cols, kind="mergesort")
        .reset_index(drop=True)
    )
    team_totals["team_cum"] = (
        team_totals.groupby([team_col, "season"])["Team Total Points GW"].cumsum()
    )
    team_totals["team_cum_prior"] = (
        team_totals["team_cum"] - team_totals["Team Total Points GW"]
    )

    out = out.merge(
        team_totals[tgw_cols + ["team_cum_prior"]],
        on=tgw_cols, how="left",
    )

    # Step 2: Player cumulative — sum per-player points across
    # (UUID, season, GW), then cumsum in GW order, then shift.
    player_gw = (
        out.groupby(["Player UUID", "season", "Gameweek"], as_index=False)["_pts_nn"]
        .sum()
        .rename(columns={"_pts_nn": "_pts_player_gw"})
        .sort_values(["Player UUID", "season", "Gameweek"], kind="mergesort")
        .reset_index(drop=True)
    )
    player_gw["player_cum"] = (
        player_gw.groupby(["Player UUID", "season"])["_pts_player_gw"].cumsum()
    )
    player_gw["player_cum_prior"] = (
        player_gw["player_cum"] - player_gw["_pts_player_gw"]
    )

    out = out.merge(
        player_gw[["Player UUID", "season", "Gameweek", "player_cum_prior"]],
        on=["Player UUID", "season", "Gameweek"], how="left",
    )

    # Step 3: Compute the causal ratio.
    with np.errstate(divide="ignore", invalid="ignore"):
        causal_pct = np.where(
            out["team_cum_prior"] > 0,
            out["player_cum_prior"] / out["team_cum_prior"] * 100.0,
            0.0,
        )
    # Clip to [0, 100] — a player cannot by construction account for more
    # than their team's cumulative points, but floating-point or rare
    # edge cases (e.g. rows where the player's team changed mid-season
    # and the team total has temporary dips) are bounded here defensively.
    out["Team Causal Contribution Pct"] = np.round(
        np.clip(causal_pct, 0.0, 100.0), 2
    )

    # --- Cleanup: drop scratch columns.
    out = out.drop(columns=["_pts_nn", "team_cum_prior", "player_cum_prior"])

    log.info("Team-contribution features computed.")
    return out.reset_index(drop=True)

In [128]:
"""Section 4.9 — Ghost-player filter."""


def filter_ghost_players(df: pd.DataFrame, season: str = CURRENT_SEASON) -> pd.DataFrame:
    """Remove current-season players with zero aggregate minutes.

    Historical rows are not touched. A player is classified a ghost iff
    their total ``Minutes Played`` across the current season is zero;
    all of their current-season rows (and only those) are then dropped.

    Parameters
    ----------
    df : pandas.DataFrame
        Combined frame.
    season : str
        Season identifier to restrict the filter to.

    Returns
    -------
    pandas.DataFrame
        Filtered frame.
    """
    current_mask = df["season"] == season
    current = df[current_mask]
    if current.empty:
        return df

    total_minutes_by_player = current.groupby("Player UUID")["Minutes Played"].sum()
    active_uuids = total_minutes_by_player[total_minutes_by_player > 0].index

    n_before = len(df)
    out = df[~current_mask | df["Player UUID"].isin(active_uuids)].copy()
    n_dropped = n_before - len(out)
    log.info(
        "Ghost filter: removed %s current-season rows "
        "(from %s players with zero aggregate minutes).",
        f"{n_dropped:,}",
        int((total_minutes_by_player == 0).sum()),
    )
    return out.reset_index(drop=True)

In [129]:
"""Section 4.10 — Final projection and persistence."""

BASE_COLUMNS: tuple[str, ...] = (
    "Player UUID", "Code", "Player Name", "Web Name",
    "Player Team ID", "Player Team Name",
    "season", "Gameweek", "Fixture Index",
    "Minutes Played", "Goals Scored", "Assists", "Clean Sheet",
    "Goals Conceded", "Yellow Card", "Red Cards", "Total Points",
    "Threat", "ICT Index", "Influence", "Creativity",
    "Opponent ID", "Opponent Name", "Opponent Difficulty", "Is Home",
    "Position", "Injury/Unavailable",
)

CONTRIBUTION_COLUMNS: tuple[str, ...] = (
    "Team Total Points GW",
    "Team GW Contribution Pct",
    "Team Causal Contribution Pct",
    "Team Contribution Rank GW",
)


def project_and_persist(df: pd.DataFrame, output_path: Path) -> pd.DataFrame:
    """Project to canonical column order and write the artefact."""
    base_present = [c for c in BASE_COLUMNS if c in df.columns]
    contrib_present = [c for c in CONTRIBUTION_COLUMNS if c in df.columns]
    lagged_cols = sorted(c for c in df.columns if c.startswith("Avg_"))
    final_cols = base_present + contrib_present + lagged_cols

    projected = df.loc[:, final_cols].copy()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    projected.to_csv(output_path, index=False, encoding="utf-8-sig")
    log.info(
        "Wrote %s rows × %d columns to %s.",
        f"{len(projected):,}", projected.shape[1], output_path,
    )
    return projected

In [130]:
"""Section 4.11 — Structured validation report."""


@dataclass
class ValidationReport:
    total_rows: int = 0
    seasons: list[str] = field(default_factory=list)
    current_season_rows: int = 0
    duplicate_key_rows: int = 0
    dgw_rows: int = 0
    dgw_rows_by_season: pd.Series = field(default_factory=lambda: pd.Series(dtype=int))
    missing_opponent_difficulty: int = 0
    unresolved_positions: int = 0
    null_player_uuid: int = 0
    rows_per_season: pd.Series = field(default_factory=lambda: pd.Series(dtype=int))

    def as_summary_table(self) -> pd.DataFrame:
        return pd.DataFrame(
            {
                "Value": [
                    self.total_rows,
                    ", ".join(self.seasons),
                    self.current_season_rows,
                    self.duplicate_key_rows,
                    self.dgw_rows,
                    self.missing_opponent_difficulty,
                    self.unresolved_positions,
                    self.null_player_uuid,
                ]
            },
            index=[
                "Total rows",
                "Seasons covered",
                f"{CURRENT_SEASON} rows",
                "Duplicate (UUID, season, GW, Opponent) rows",
                "Double-Gameweek rows retained (expected — see §5.3)",
                "Missing Opponent Difficulty",
                "Unresolved Position rows",
                "Null Player UUID rows",
            ],
        )


def validate_master(df: pd.DataFrame) -> ValidationReport:
    """Compute structured diagnostics over the master training set."""
    r = ValidationReport()
    r.total_rows = len(df)
    r.seasons = sorted(df["season"].unique().tolist())
    r.current_season_rows = int((df["season"] == CURRENT_SEASON).sum())
    key = ["Player UUID", "season", "Gameweek", "Opponent ID"]
    r.duplicate_key_rows = int(df.duplicated(subset=key, keep=False).sum())
    if "Fixture Index" in df.columns:
        dgw_mask = df["Fixture Index"] > 1
        r.dgw_rows = int(dgw_mask.sum())
        r.dgw_rows_by_season = df[dgw_mask].groupby("season").size().rename("dgw_rows")
    r.missing_opponent_difficulty = int(df["Opponent Difficulty"].isna().sum())
    r.unresolved_positions = int(df["Position"].isna().sum())
    r.null_player_uuid = int(df["Player UUID"].isna().sum())
    r.rows_per_season = df.groupby("season").size().rename("rows")
    return r

### 4.12 Pipeline Orchestration

Composes the API acquisition, current-season synthesis, merge,
contribution computation, filtering, feature engineering and
persistence steps into a single linear flow.

In [131]:
"""Section 4.12 — Orchestration."""

# 1. Load frozen historical baseline (Notebook 1 output).
if not HISTORICAL_BASELINE_PATH.exists():
    raise FileNotFoundError(
        f"Historical baseline not found at {HISTORICAL_BASELINE_PATH}. "
        "Run historical_baseline.ipynb first."
    )
historical_frame = pd.read_csv(HISTORICAL_BASELINE_PATH, encoding="utf-8-sig")
log.info("Loaded historical baseline: %s rows.", f"{len(historical_frame):,}")

# 2. Acquire and synthesise the current season (§4.1 – §4.6).
current_frame = build_current_season_frame()

# 3. Merge with overlap resolution (§4.7).
combined = merge_with_historical(historical_frame, current_frame)

# 4. Recompute Fixture Index over the combined frame so that every row's
#    index reflects its position within the final combined (UUID, season,
#    GW) group, not the partition it originated from.
combined = combined.sort_values(
    ["Player UUID", "season", "Gameweek", "Opponent ID"], kind="mergesort"
)
combined["Fixture Index"] = (
    combined.groupby(["Player UUID", "season", "Gameweek"]).cumcount() + 1
).astype("Int64")

# 5. Ghost-player filter applied *before* team contributions so that
#    contribution denominators reflect only teams' active players.
combined = filter_ghost_players(combined, season=CURRENT_SEASON)

# 6. Team contributions (§4.8) — requires combined rows in memory.
combined = compute_team_contributions(combined)

# 7. Re-compute absence flag and rolling features over the combined
#    dataset so that the last gameweek(s) of the historical partition
#    inform the first gameweek(s) of the current partition.
combined = flag_consecutive_absences(combined)
combined = compute_rolling_features(combined)

# 8. Project and persist the master training set (§4.10).
master_training_set = project_and_persist(combined, MASTER_TRAINING_SET_PATH)

# 9. Diagnostics for Section 5.
report = validate_master(master_training_set)
log.info("Pipeline complete.")

15:07:34 | INFO    | Loaded historical baseline: 133,647 rows.
15:07:34 | INFO    | Fetching bootstrap-static and fixtures …
15:07:35 | INFO    | Static data written: 20 teams, 829 players, 380 fixtures.
15:07:35 | INFO    | Ingestable gameweeks (finished & data-checked): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
15:07:35 | INFO    | GW 1  fetched: 690 rows → gw1.csv
15:07:36 | INFO    | GW 2  fetched: 705 rows → gw2.csv
15:07:36 | INFO    | GW 3  fetched: 712 rows → gw3.csv
15:07:36 | INFO    | GW 4  fetched: 740 rows → gw4.csv
15:07:36 | INFO    | GW 5  fetched: 741 rows → gw5.csv
15:07:36 | INFO    | GW 6  fetched: 742 rows → gw6.csv
15:07:36 | INFO    | GW 7  fetched: 743 rows → gw7.csv
15:07:36 | INFO    | GW 8  fetched: 745 rows → gw8.csv
15:07:36 | INFO    | GW 9  fetched: 746 rows → gw9.csv
15:07:36 | INFO    | GW 10 fetched: 747 rows → gw10.csv
15:07:37 | INFO    | GW 11 fetched: 752 rows → gw11.csv


C:\Users\SOFI\AppData\Local\Temp\ipykernel_5160\1800437569.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([hist_aligned, curr_aligned], ignore_index=True)


15:07:41 | INFO    | Merge complete: 133,647 historical + 24,655 current → 158,302 combined (0 overlap rows resolved).
15:07:42 | INFO    | Ghost filter: removed 8,613 current-season rows (from 303 players with zero aggregate minutes).
15:07:43 | INFO    | Team-contribution features computed.
15:08:00 | INFO    | Wrote 149,689 rows × 49 columns to C:\Python\fpl_pipeline\data\processed\master_training_set.csv.
15:08:00 | INFO    | Pipeline complete.


## 5. Results and Validation

Headline diagnostics for the master training set produced above.

In [132]:
"""Section 5.1 — Headline summary."""
report.as_summary_table()

,Value
Total rows,149689
Seasons covered,"2020-21, 2021-22, 2022-23, 2023-24, 2024-25, 2..."
2025-26 rows,16042
"Duplicate (UUID, season, GW, Opponent) rows",0
Double-Gameweek rows retained (expected — see §5.3),6650
Missing Opponent Difficulty,0
Unresolved Position rows,0
Null Player UUID rows,0


In [133]:
"""Diagnose the row-explosion in current-season rows."""

curr = master_training_set[master_training_set["season"] == CURRENT_SEASON].copy()

# How many rows per (player, gameweek)?
rows_per_player_gw = curr.groupby(["Player UUID", "Gameweek"]).size()
print("Rows per (player, GW) in current season:")
print(rows_per_player_gw.value_counts().sort_index())
print()

# How many distinct players per GW?
players_per_gw = curr.groupby("Gameweek")["Player UUID"].nunique()
print("Distinct players per GW in current season:")
print(players_per_gw)
print()

# Rows per GW total
rows_per_gw = curr.groupby("Gameweek").size()
print("Total rows per GW in current season:")
print(rows_per_gw)
print()

# Sample a player who has an unusually high number of rows in one GW
problem_players = rows_per_player_gw[rows_per_player_gw > 2]
if len(problem_players) > 0:
    print(f"Players with >2 rows in a single GW: {len(problem_players)}")
    print("Sample of problematic groups:")
    sample_idx = problem_players.head(3).index
    for uuid_val, gw_val in sample_idx:
        sample = curr[(curr["Player UUID"] == uuid_val) & (curr["Gameweek"] == gw_val)]
        display(sample[[
            "Player Name", "Player Team Name", "Gameweek", "Fixture Index",
            "Opponent Name", "Opponent ID", "Opponent Difficulty",
            "Minutes Played", "Total Points", "Is Home"
        ]])

# Missing Opponent Difficulty — what rows are these?
print("\nRows with missing Opponent Difficulty:")
missing_od = curr[curr["Opponent Difficulty"].isna()]
print(f"Count: {len(missing_od)}")
if len(missing_od) > 0:
    display(missing_od[[
        "Player Name", "Player Team Name", "Gameweek",
        "Opponent Name", "Opponent ID"
    ]].head(10))

Rows per (player, GW) in current season:
1    15938
2       52
Name: count, dtype: int64

Distinct players per GW in current season:
Gameweek
1     453
2     463
3     468
4     493
5     493
6     493
7     493
8     495
9     495
10    496
11    498
12    500
13    500
14    501
15    501
16    501
17    503
18    504
19    504
20    506
21    510
22    511
23    513
24    517
25    523
26    523
27    523
28    523
29    523
30    523
31    418
32    523
Name: Player UUID, dtype: int64

Total rows per GW in current season:
Gameweek
1     453
2     463
3     468
4     493
5     493
6     493
7     493
8     495
9     495
10    496
11    498
12    500
13    500
14    501
15    501
16    501
17    503
18    504
19    504
20    506
21    510
22    511
23    513
24    517
25    523
26    575
27    523
28    523
29    523
30    523
31    418
32    523
dtype: int64


Rows with missing Opponent Difficulty:
Count: 0


In [134]:
"""Diagnose the inflated DGW count."""

# How many (UUID, season, GW) groups have > 1 row?
group_sizes = (
    master_training_set
    .groupby(["Player UUID", "season", "Gameweek"])
    .size()
)
print("Distribution of rows per (UUID, season, GW) group:")
print(group_sizes.value_counts().sort_index())
print()

# Break down by season
multi_row_groups = group_sizes[group_sizes > 1].reset_index(name="n_rows")
print("Multi-row groups by season:")
print(multi_row_groups.groupby("season")["n_rows"].agg(["count", "sum"]))
print()

# Distribution of Fixture Index values per season
print("Fixture Index distribution by season:")
fx_dist = (
    master_training_set
    .groupby("season")["Fixture Index"]
    .value_counts()
    .unstack(fill_value=0)
)
print(fx_dist)
print()

# What does a 'Fixture Index > 1' row look like? Pick 5 random ones
fi_gt_1 = master_training_set[master_training_set["Fixture Index"] > 1]
print(f"\nSample of rows with Fixture Index > 1:")
display(fi_gt_1[[
    "Player Name", "season", "Gameweek", "Fixture Index",
    "Opponent Name", "Minutes Played", "Total Points"
]].head(10))

# Are these rows genuine DGWs (same player, same GW, multiple opponents)?
sample_group_keys = fi_gt_1.head(5)[["Player UUID", "season", "Gameweek"]].drop_duplicates()
for _, row in sample_group_keys.iterrows():
    sub = master_training_set[
        (master_training_set["Player UUID"] == row["Player UUID"]) &
        (master_training_set["season"] == row["season"]) &
        (master_training_set["Gameweek"] == row["Gameweek"])
    ]
    print(f"\nGroup: {row['Player UUID'][:8]}..., {row['season']}, GW {row['Gameweek']}")
    display(sub[["Player Name", "Opponent Name", "Fixture Index", "Minutes Played"]])

Distribution of rows per (UUID, season, GW) group:
1    136428
2      6572
3        39
Name: count, dtype: int64

Multi-row groups by season:
         count   sum
season              
2020-21   1437  2913
2021-22   2217  4434
2022-23   1548  3096
2023-24    983  1966
2024-25    374   748
2025-26     52   104

Fixture Index distribution by season:
Fixture Index      1     2   3
season                        
2020-21        22889  1437  39
2021-22        23230  2217   0
2022-23        24957  1548   0
2023-24        28742   983   0
2024-25        27231   374   0
2025-26        15990    52   0


Sample of rows with Fixture Index > 1:


,Player Name,season,Gameweek,Fixture Index,Opponent Name,Minutes Played,Total Points
18,Filip Benkovic,2020-21,19,2,Southampton,0,0
26,Filip Benkovic,2020-21,26,2,Burnley,0,0
35,Filip Benkovic,2020-21,35,2,Newcastle,0,0
71,Bryan Gil Salvatierra,2023-24,35,2,Chelsea,12,1
74,Bryan Gil Salvatierra,2023-24,37,2,Man City,0,0
100,Deniz Undav,2022-23,27,2,Leeds,0,0
102,Deniz Undav,2022-23,29,2,Brentford,7,4
107,Deniz Undav,2022-23,34,2,Wolves,79,12
110,Deniz Undav,2022-23,36,2,Newcastle,84,3
112,Deniz Undav,2022-23,37,2,Southampton,16,1



Group: 00015d7a..., 2020-21, GW 19


,Player Name,Opponent Name,Fixture Index,Minutes Played
17,Filip Benkovic,Chelsea,1,0
18,Filip Benkovic,Southampton,2,0



Group: 00015d7a..., 2020-21, GW 26


...

,Player Name,Opponent Name,Fixture Index,Minutes Played
34,Filip Benkovic,Man Utd,1,0
35,Filip Benkovic,Newcastle,2,0



Group: 00144381..., 2023-24, GW 35


,Player Name,Opponent Name,Fixture Index,Minutes Played
70,Bryan Gil Salvatierra,Arsenal,1,0
71,Bryan Gil Salvatierra,Chelsea,2,12



Group: 00144381..., 2023-24, GW 37


,Player Name,Opponent Name,Fixture Index,Minutes Played
73,Bryan Gil Salvatierra,Burnley,1,0
74,Bryan Gil Salvatierra,Man City,2,0


In [135]:
"""Section 5.2 — Records per season."""
records_table = report.rows_per_season.to_frame()
records_table["share_%"] = (
    records_table["rows"] / records_table["rows"].sum() * 100
).round(2)
records_table

,rows,share_%
season,,
2020-21,24365,16.28
2021-22,25447,17.00
2022-23,26505,17.71
2023-24,29725,19.86
2024-25,27605,18.44
2025-26,16042,10.72


In [136]:
"""Section 5.3 — Double-Gameweek distribution by season.

DGWs occur when a team plays two matches within a single gameweek —
a scheduling artefact that became especially prevalent during the
2020-21 and 2021-22 COVID-disrupted seasons (fixture compression
following postponements) and the 2022-23 World Cup compression.
The figures below should track real-world FPL history; any gross
deviation would indicate a bug in the fixture-derived team
identification pipeline.
"""
_dgw_frame = master_training_set[master_training_set["Fixture Index"] > 1]

dgw_table = pd.DataFrame({
    "dgw_rows": _dgw_frame.groupby("season").size(),
    "dgw_groups": (
        _dgw_frame
        .drop_duplicates(subset=["Player UUID", "season", "Gameweek"])
        .groupby("season")
        .size()
    ),
})
dgw_table["avg_rows_per_group"] = (
    dgw_table["dgw_rows"] / dgw_table["dgw_groups"]
).round(2)
dgw_table

,dgw_rows,dgw_groups,avg_rows_per_group
season,,,
2020-21,1476,1437,1.03
2021-22,2217,2217,1.00
2022-23,1548,1548,1.00
2023-24,983,983,1.00
2024-25,374,374,1.00
2025-26,52,52,1.00


In [137]:
"""Section 5.4 — Team contribution metric distributions.

Sanity checks on the three contribution features:

1. GW% entries within an active team-gameweek must sum to ~100.
2. Causal% must lie in [0, 100] for every row.
3. Contribution Rank must lie in (0, 1].
"""
team_col = "Player Team ID"

# GW% sum check
sum_check = (
    master_training_set
    .groupby([team_col, "season", "Gameweek"])["Team GW Contribution Pct"].sum()
    .reset_index(name="sum_pct")
)
active_sums = sum_check[sum_check["sum_pct"] > 0]
out_of_band_gw = active_sums[(active_sums["sum_pct"] < 99.0) | (active_sums["sum_pct"] > 101.0)]

# Causal% range check
causal_out_of_range = master_training_set[
    (master_training_set["Team Causal Contribution Pct"] < 0)
    | (master_training_set["Team Causal Contribution Pct"] > 100)
]

# Rank range check
rank_out_of_range = master_training_set[
    (master_training_set["Team Contribution Rank GW"] <= 0)
    | (master_training_set["Team Contribution Rank GW"] > 1)
]

print(f"Active team-gameweek groups            : {len(active_sums):,}")
print(f"  GW% sums in [99, 101]                : {len(active_sums) - len(out_of_band_gw):,}")
print(f"  GW% sums out of band                 : {len(out_of_band_gw):,}")
print()
print(f"Causal% rows out of [0, 100]           : {len(causal_out_of_range):,}")
print(f"Rank rows out of (0, 1]                : {len(rank_out_of_range):,}")
print()

master_training_set[
    ["Team GW Contribution Pct", "Team Causal Contribution Pct", "Team Contribution Rank GW"]
].describe().round(3)

Active team-gameweek groups            : 4,242
  GW% sums in [99, 101]                : 4,242
  GW% sums out of band                 : 0

Causal% rows out of [0, 100]           : 0
Rank rows out of (0, 1]                : 0



,Team GW Contribution Pct,Team Causal Contribution Pct,Team Contribution Rank GW
count,149689.000,149689.000,149689.000
mean,2.834,2.891,0.321
std,4.996,3.435,0.157
min,0.000,0.000,0.008
25%,0.000,0.000,0.194
50%,0.000,1.520,0.353
75%,4.170,5.080,0.433
max,48.480,41.670,0.810


In [138]:
"""Section 5.4 — Top 10 season contributors (causal cumulative share).

End-of-season snapshot: for every (player, season) take the final-row
causal cumulative contribution, then sort. A defensible dataset should
recover canonical team-leading players in each season.
"""
end_of_season = (
    master_training_set
    .sort_values(["Player UUID", "season", "Gameweek"])
    .groupby(["Player UUID", "season"], as_index=False)
    .tail(1)
)
top_contrib = (
    end_of_season
    .loc[:, ["Player Name", "Player Team Name", "season",
             "Team Causal Contribution Pct", "Total Points"]]
    .sort_values("Team Causal Contribution Pct", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_contrib

,Player Name,Player Team Name,season,Team Causal Contribution Pct,Total Points
0,Mohamed Salah,Liverpool,2024-25,16.21,10
1,Cole Palmer,Chelsea,2023-24,15.34,6
2,Harry Kane,Spurs,2022-23,15.20,16
3,Mohamed Salah,Liverpool,2022-23,14.35,5
4,Heung-Min Son,Spurs,2021-22,13.93,12
5,Bruno Miguel Borges Fernandes,Man Utd,2020-21,13.68,0
6,Harry Kane,Spurs,2020-21,13.46,10
7,Bryan Mbeumo,Brentford,2024-25,13.44,8
8,Mohamed Salah,Liverpool,2020-21,13.34,6
9,Son Heung-min,Spurs,2023-24,13.21,7


In [139]:
"""Diagnose team contribution computation."""

# 1. Which grouping column did the function use?
print(f"Player Team ID in master?  {'Player Team ID' in master_training_set.columns}")
print(f"Player Team Name in master? {'Player Team Name' in master_training_set.columns}")

# 2. How many unique team names exist per season?
print("\nUnique Player Team Name per season:")
print(master_training_set.groupby("season")["Player Team Name"].nunique())

# 3. Are there Player Team Name values that appear in only one season?
team_season_counts = (
    master_training_set
    .groupby("Player Team Name")["season"]
    .nunique()
    .sort_values()
)
print(f"\nTeam names with inconsistent season coverage:")
print(team_season_counts.head(15))

# 4. How many rows have missing Player Team Name?
missing_team = master_training_set["Player Team Name"].isna().sum()
print(f"\nRows with missing Player Team Name: {missing_team}")

# 5. Sanity: what's a real top player's contribution?
# Pick Mohamed Salah, 2023-24, last row
salah = master_training_set[
    (master_training_set["Player Name"].str.contains("Salah", na=False, case=False))
    & (master_training_set["season"] == "2023-24")
].sort_values("Gameweek").tail(1)
print("\nSalah 2023-24 final row:")
if len(salah) > 0:
    display(salah[[
        "Player Name", "Player Team Name", "Gameweek",
        "Total Points", "Team Total Points GW",
        "Team GW Contribution Pct", "Team Causal Contribution Pct"
    ]])

# 6. Spot-check: one team, one gameweek, sum of GW contributions
sample = master_training_set[
    (master_training_set["season"] == "2023-24")
    & (master_training_set["Gameweek"] == 34)
    & (master_training_set["Player Team Name"] == "Liverpool")
]
print(f"\nLiverpool GW34 2023-24: {len(sample)} rows")
if len(sample) > 0:
    total_team_points = sample["Total Points"].sum()
    total_gw_pct = sample["Team GW Contribution Pct"].sum()
    print(f"  Team total points (computed from rows): {total_team_points}")
    print(f"  Sum of Team GW Contribution Pct: {total_gw_pct:.2f}")
    print(f"  Team Total Points GW (from feature col): {sample['Team Total Points GW'].iloc[0]}")

Player Team ID in master?  True
Player Team Name in master? True

Unique Player Team Name per season:


season
2020-21    20
2021-22    20
2022-23    20
2023-24    20
2024-25    20
2025-26    20
Name: Player Team Name, dtype: int64

Team names with inconsistent season coverage:
Player Team Name
Luton            1
Ipswich          1
Sunderland       1
Norwich          1
Watford          1
West Brom        1
Sheffield Utd    2
Bournemouth      4
Leeds            4
Southampton      4
Nott'm Forest    4
Burnley          4
Leicester        4
Brentford        5
Fulham           5
Name: season, dtype: int64

Rows with missing Player Team Name: 0

Salah 2023-24 final row:


,Player Name,Player Team Name,Gameweek,Total Points,Team Total Points GW,Team GW Contribution Pct,Team Causal Contribution Pct
44863,Mohamed Salah,Liverpool,38,6,74.0,8.11,11.24



Liverpool GW34 2023-24: 42 rows
  Team total points (computed from rows): 59
  Sum of Team GW Contribution Pct: 99.96
  Team Total Points GW (from feature col): 59.0


In [140]:
"""Section 5.5 — Final assertion gate.

Integrity invariants that must hold for the master training set.
"""
assert report.duplicate_key_rows == 0, (
    f"Found {report.duplicate_key_rows} rows duplicated on "
    f"(Player UUID, season, Gameweek, Opponent ID)."
)
assert report.null_player_uuid == 0, (
    f"Found {report.null_player_uuid} rows with a null Player UUID."
)
assert report.unresolved_positions == 0, (
    f"Found {report.unresolved_positions} rows with an unresolved Position."
)

# GW contribution sum within ±1pp of 100 for active team-gameweeks.
assert len(out_of_band_gw) == 0, (
    f"Found {len(out_of_band_gw)} team-gameweek groups whose GW% sum "
    f"deviates by more than 1pp from 100."
)

# Causal contribution must lie in [0, 100] by construction.
assert len(causal_out_of_range) == 0, (
    f"Found {len(causal_out_of_range)} rows with a Team Causal "
    f"Contribution Pct outside [0, 100]."
)

# Fixture Index must be contiguous 1..N within each (UUID, season, GW) group.
fx_check = (
    master_training_set
    .groupby(["Player UUID", "season", "Gameweek"])["Fixture Index"]
    .agg(lambda s: list(sorted(s)) == list(range(1, len(s) + 1)))
)
assert fx_check.all(), (
    f"Found {(~fx_check).sum()} (player, season, GW) groups whose "
    f"Fixture Index is not a contiguous 1..N sequence."
)

log.info("All integrity assertions passed.")

15:08:04 | INFO    | All integrity assertions passed.


## 6. Conclusion and Downstream Use

### 6.1 Artefact Summary

The notebook produces `data/processed/master_training_set.csv`, a
single CSV whose primary key is the tuple
$(\text{Player UUID},\ \text{season},\ \text{Gameweek},\ \text{Opponent ID})$.
The schema is the union of the historical baseline's schema, the
four team-contribution features defined in §2.6, and the lagged
rolling features `Avg_*_L{3,5}` re-computed over the combined
dataset.

### 6.2 Known Limitations

- **API snapshot semantics.** Each invocation of this notebook
  captures the FPL platform's state at that instant. Minor
  post-match revisions (e.g. bonus-point adjustments, disciplinary
  reclassifications) may alter rows retroactively; rerunning the
  notebook after such a revision is benign and deterministic.
- **Ghost filter is current-season only.** If the historical
  baseline contained zero-minute players who later became active
  in the current season, those historical ghost rows are preserved
  by design. Removing them would rewrite the frozen input of
  Notebook 1.
- **Contribution features depend on team identity.** Where
  `Player Team ID` is unavailable (e.g. rare current-season rows
  whose fixture join failed), the function falls back to
  `Player Team Name`; this is cosmetically equivalent but more
  fragile in the rare case of intra-season team renames.

### 6.3 Downstream Handoff

The artefact is consumed by **`forecasting_and_backtest.ipynb`**,
which performs feature selection, time-series cross-validated model
training, the auto-substitution and captaincy backtest, and
out-of-sample RMSE/MAE reporting.

### 6.4 Reproducibility Notes

The notebook is fully deterministic given a fixed FPL API response.
Because the API is live, two runs on different days will naturally
produce different row counts as additional gameweeks become
data-checked — this is the intended behaviour. For frozen snapshots
(e.g. thesis-defense reproduction), the entire
`data/{CURRENT_SEASON}/` directory should be committed alongside the
notebook.

## References

[1] Fantasy Premier League, "FPL API — bootstrap-static, fixtures, and
    event endpoints," Fantasy Premier League, London, UK, 2025.
    [Online]. Available: https://fantasy.premierleague.com/api/

[2] A. Vaastav, "Fantasy Premier League Historical Data," GitHub
    repository, 2024. [Online]. Available:
    https://github.com/vaastav/Fantasy-Premier-League

## 6. Modeling-Ready Derivative: Zero-Minute Filter

The master training set preserves every fixture-player observation for audit integrity, including rows where the player was an unused substitute or did not make the matchday squad. While valuable for completeness, these rows introduce two modelling pathologies:

1. **Distributional distortion.** Zero-minute rows collapse the target variable `Total Points` to deterministic values (typically 0 or −1 for disciplinary deductions), artificially inflating the "blank" tier of the score distribution. A regressor trained on this mixture effectively learns to predict zero for any player with a recent DNP streak, masking the true signal in the participating subset.

2. **Rolling feature contamination.** Features such as `Avg_Total Points_L5` or `Avg_Minutes Played_L5` treat a zero-minute row identically to a legitimately poor performance. This flattens form signals: a previously in-form player who is rotated for one week appears to have collapsed statistically, and the model over-corrects downward on their next fixture.

This section produces `master_training_set_no0min.csv`, a modelling derivative where all rows with `Minutes Played == 0` are removed. The original `master_training_set.csv` remains untouched and authoritative for audit purposes. This separation of concerns — canonical unfiltered ledger vs. modelling-ready derivative — follows the principle of dataset versioning recommended by [Gebru et al., 2021][^datasheets].

[^datasheets]: T. Gebru *et al.*, "Datasheets for Datasets," *Communications of the ACM*, vol. 64, no. 12, pp. 86–92, Dec. 2021, doi: 10.1145/3458723.

### 6.1 Filter Specification

**Inclusion criterion:** `Minutes Played > 0`

**Justification:** Any match participation, however brief (even a 1-minute substitution appearance), represents a legitimate observation of the player's point-scoring context. Only true non-participation events are excluded. This matches the convention used by published FPL analytics pipelines [^fplreview].

[^fplreview]: M. Amosnjr, "Mathematical Modelling for Fantasy Premier League," *MSc Thesis, University of Manchester*, 2020, Section 3.2.

### 6.2 Implementation

In [141]:
# =============================================================================
# §6.2 — Build modeling-ready derivative by removing zero-minute rows
# =============================================================================
# Input:  master_training_set.csv (canonical, preserved)
# Output: master_training_set_no0min.csv (modeling-ready derivative)
# =============================================================================

log.info("=" * 70)
log.info("§6 — Building zero-minute-filtered modeling derivative")
log.info("=" * 70)

# -- Paths --------------------------------------------------------------------
MASTER_TRAINING_SET_PATH   = PROCESSED_DIR / "master_training_set.csv"
MODELING_DERIVATIVE_PATH   = PROCESSED_DIR / "master_training_set_no0min.csv"

# Sanity check: canonical master must exist from §5
if not MASTER_TRAINING_SET_PATH.exists():
    raise FileNotFoundError(
        f"Canonical master training set not found at {MASTER_TRAINING_SET_PATH}. "
        f"Re-run §5 (Merge, Validate, Persist) before producing the derivative."
    )

# -- Load canonical ------------------------------------------------------------
log.info("Loading canonical master training set from %s", MASTER_TRAINING_SET_PATH)
_master_df = pd.read_csv(MASTER_TRAINING_SET_PATH)
_n_total = len(_master_df)
log.info("Canonical row count: %s", f"{_n_total:,}")

# -- Pre-filter diagnostics ----------------------------------------------------
# Report the distribution of Minutes Played to justify the threshold choice
_mp_col = "Minutes Played"
if _mp_col not in _master_df.columns:
    raise KeyError(
        f"Expected column '{_mp_col}' not found in master_training_set.csv. "
        f"Available columns: {sorted(_master_df.columns.tolist())}"
    )

_zero_min_mask   = _master_df[_mp_col] == 0
_n_zero_min      = int(_zero_min_mask.sum())
_pct_zero        = 100.0 * _n_zero_min / _n_total

_by_season_before = (
    _master_df.assign(is_zero=_zero_min_mask)
              .groupby("season", observed=True)["is_zero"]
              .agg(total="count", zero="sum")
              .assign(kept=lambda d: d["total"] - d["zero"],
                      pct_removed=lambda d: (100.0 * d["zero"] / d["total"]).round(2))
              .reset_index()
)

log.info("")
log.info("Zero-minute row distribution by season (pre-filter):")
log.info("\n%s", _by_season_before.to_string(index=False))
log.info("")
log.info("Total zero-minute rows:  %s  (%.2f%% of canonical)", f"{_n_zero_min:,}", _pct_zero)
log.info("Total retained rows:     %s", f"{_n_total - _n_zero_min:,}")

# -- Apply filter --------------------------------------------------------------
_modeling_df = _master_df.loc[~_zero_min_mask].copy()

# -- Integrity assertions ------------------------------------------------------
assert len(_modeling_df) == _n_total - _n_zero_min, (
    f"Row accounting mismatch: "
    f"expected {_n_total - _n_zero_min}, got {len(_modeling_df)}"
)
assert (_modeling_df[_mp_col] > 0).all(), (
    "Post-filter: at least one row still has Minutes Played == 0"
)
assert _modeling_df["Player UUID"].isna().sum() == 0, (
    "Post-filter: null UUIDs detected — this should never happen"
)

# The filter should preserve all seasons and not collapse any to empty
_seasons_before = set(_master_df["season"].unique())
_seasons_after  = set(_modeling_df["season"].unique())
assert _seasons_after == _seasons_before, (
    f"Post-filter: seasons lost. Before={sorted(_seasons_before)}, "
    f"After={sorted(_seasons_after)}"
)

# The filter should preserve DGW row structure (composite key uniqueness)
_dup_check = (
    _modeling_df.groupby(
        ["Player UUID", "season", "Gameweek", "Opponent ID"]
    ).size()
)
assert (_dup_check == 1).all(), (
    f"Post-filter: composite key duplicates detected "
    f"({int((_dup_check > 1).sum())} groups affected). "
    f"This indicates an upstream DGW dedup bug."
)

# -- Post-filter diagnostics ---------------------------------------------------
_by_season_after = (
    _modeling_df.groupby("season", observed=True)
                .agg(rows=("Player UUID", "size"),
                     players=("Player UUID", "nunique"),
                     min_gw=("Gameweek", "min"),
                     max_gw=("Gameweek", "max"),
                     mean_pts=("Total Points", "mean"),
                     median_pts=("Total Points", "median"))
                .round(3)
                .reset_index()
)

log.info("")
log.info("Modeling derivative summary by season:")
log.info("\n%s", _by_season_after.to_string(index=False))

# Distribution shift: before vs after mean points (expected to rise)
_mean_before = _master_df["Total Points"].mean()
_mean_after  = _modeling_df["Total Points"].mean()
log.info("")
log.info("Mean Total Points — canonical:    %.3f", _mean_before)
log.info("Mean Total Points — modeling:     %.3f  (Δ = %+.3f)",
         _mean_after, _mean_after - _mean_before)
log.info(
    "Distributional shift confirms filter effect: non-participant rows "
    "(deterministic zeros) are removed, raising the conditional mean of the "
    "retained observations. This is the desired behaviour."
)

# -- Atomic persist ------------------------------------------------------------
log.info("")
log.info("Writing modeling derivative to %s", MODELING_DERIVATIVE_PATH)

_tmp_path = MODELING_DERIVATIVE_PATH.with_suffix(".csv.tmp")
_modeling_df.to_csv(_tmp_path, index=False, encoding="utf-8")
_tmp_path.replace(MODELING_DERIVATIVE_PATH)   # atomic rename

# Re-read and verify persisted artifact matches in-memory derivative
_verify_df = pd.read_csv(MODELING_DERIVATIVE_PATH)
assert len(_verify_df) == len(_modeling_df), (
    f"Persisted row count mismatch: "
    f"in-memory={len(_modeling_df):,}, on-disk={len(_verify_df):,}"
)
assert list(_verify_df.columns) == list(_modeling_df.columns), (
    "Persisted column order mismatch"
)

log.info("Persisted %s rows × %s columns to %s",
         f"{len(_modeling_df):,}",
         len(_modeling_df.columns),
         MODELING_DERIVATIVE_PATH.name)
log.info("")
log.info("§6 complete. Modeling derivative is ready for consumption by Notebook 3.")
log.info("=" * 70)

23:12:00 | INFO    | ======================================================================
23:12:00 | INFO    | §6 — Building zero-minute-filtered modeling derivative
23:12:00 | INFO    | ======================================================================
23:12:00 | INFO    | Loading canonical master training set from C:\Python\fpl_pipeline\data\processed\master_training_set.csv
23:12:02 | INFO    | Canonical row count: 149,689
23:12:02 | INFO    | 
23:12:02 | INFO    | Zero-minute row distribution by season (pre-filter):
23:12:02 | INFO    | 
 season  total  zero  kept  pct_removed
2020-21  24365 13972 10393        57.34
2021-22  25447 14962 10485        58.80
2022-23  26505 15160 11345        57.20
2023-24  29725 18341 11384        61.70
2024-25  27605 16039 11566        58.10
2025-26  16042  6386  9656        39.81
23:12:02 | INFO    | 
23:12:02 | INFO    | Total zero-minute rows:  84,860  (56.69% of canonical)
23:12:02 | INFO    | Total retained rows:     64,829
23:12:02 | INFO

### 6.3 Relationship Between the Two Artifacts

The canonical and modeling derivative artifacts serve distinct, non-overlapping purposes:

| Artifact | Purpose | Consumer | Filter Applied |
|---|---|---|---|
| `master_training_set.csv` | Canonical audit ledger | Diagnostic tooling, integrity tests, reproducibility validation | None (every fixture-player row preserved) |
| `master_training_set_no0min.csv` | Modeling-ready derivative | `forecasting_and_backtest.ipynb` (Notebook 3) | `Minutes Played > 0` |

The filter is **one-way and idempotent**: the derivative can always be reconstructed from the canonical, but not vice versa. This asymmetry is intentional — it prevents the modeling decisions embedded in the filter from propagating upstream into the audit record. If a future modelling revision requires a different inclusion criterion (e.g., `Minutes Played ≥ 15` to exclude token appearances), the canonical remains available as the unambiguous ground truth from which any new derivative can be produced.